In [ ]:
import sys, subprocess
def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *pkgs])

pipq("scikit-learn", "pandas", "numpy", "matplotlib", "seaborn")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.max_columns", 60)
np.random.seed(0)
print("pandas", pd.__version__)

In [ ]:
df = pd.read_csv('data.csv', encoding='ISO-8859-1')
df["CustomerID"] = df["CustomerID"].astype(dtype="Int64")
display(df.head(3))
print("shape:", df.shape)
display(df.describe())
df.info()
df.isnull().sum()

In [ ]:
print("distinct country:", df["Country"].nunique())
print(df["Country"].value_counts())

print("distinct stockcode:", df["StockCode"].nunique())
print(df["StockCode"].value_counts())

print("distinct description:", df["Description"].nunique())
print(df["Description"].value_counts())

print(df["StockCode"].value_counts(dropna=False)) # StockCode and Description are the Product code/name
print(df["Description"].value_counts(dropna=False))

In [ ]:
display(df.describe())

print("Num of negative Quantity: ", df[df["Quantity"] < 0].shape[0])
print("Num of negative or zero UnitPrice: ", df[df["UnitPrice"] <= 0].shape[0])


In [ ]:
region_lookup_table = {
    "United Kingdom": "UK&IE",
    "Germany": "Western Europe",
    "France": "Western Europe",
    "EIRE": "UK&IE",
    "Spain": "Southern Europe",
    "Netherlands": "Western Europe",
    "Belgium": "Western Europe",
    "Switzerland": "Western Europe",
    "Portugal": "Southern Europe",
    "Australia": "Oceania",
    "Norway": "Northern Europe",
    "Italy": "Southern Europe",
    "Channel Islands": "UK&IE",
    "Finland": "Northern Europe",
    "Cyprus": "Southern Europe",
    "Sweden": "Northern Europe",
    "Unspecified": "Unknown",
    "Austria": "Western Europe",
    "Denmark": "Northern Europe",
    "Japan": "East Asia",
    "Poland": "Eastern Europe",
    "Israel": "Middle East",
    "USA": "North America",
    "Hong Kong": "East Asia",
    "Singapore": "Southeast Asia",
    "Iceland": "Northern Europe",
    "Canada": "North America",
    "Greece": "Southern Europe",
    "Malta": "Southern Europe",
    "United Arab Emirates": "Middle East",
    "European Community": "Europe",
    "RSA": "Africa",
    "Lebanon": "Middle East",
    "Lithuania": "Eastern Europe",
    "Brazil": "South America",
    "Czech Republic": "Eastern Europe",
    "Bahrain": "Middle East",
    "Saudi Arabia": "Middle East"
}

In [ ]:
# display(df.iloc[0:3])
# df["LineItem"] = df.index
# display(df.loc[:, ["Quantity", "UnitPrice"]].head())
# df2 = df.set_index(df.index)
# display(df2.loc[0:3])

display(df.iloc[0:3])
df.index = pd.Index([f"L{i:06d}" for i in range(len(df))], name="line-item")
display(df.loc["L000005"])

In [ ]:
print("Cancellations:", df[df["InvoiceNo"].str.startswith("C")].shape[0])
print("Num of negative or zero Quantity: ", df[df["Quantity"] <= 0].shape[0])
print("Num of negative or zero UnitPrice: ", df[df["UnitPrice"] <= 0].shape[0]) # I think I did it previuosly, but I'll just follow instruction


In [ ]:
df["Revenue"] = df["Quantity"] * df["UnitPrice"]
df["Revenue"] = df["Revenue"].round(3)
sorted_in_largest_bulk_orders = df.sort_values("Quantity", ascending=False)
print("Largest Bulk Orders: \n", sorted_in_largest_bulk_orders.head(1))
sorted_in_largest_returns = df.sort_values("Revenue", ascending=False)
print("Largest returns: \n", sorted_in_largest_returns.head(1))

In [ ]:
# Begin of Phoenix
data = df

In [ ]:
# Phase C
# 3.6
data["Country"] = data["Country"].replace({ "EIRE": "Ireland", "RSA": "South Africa", "Unspecified": pd.NA })  # type: ignore
print("EIRE -> Ireland, RSA -> South Africa, Unspecified -> missing")
print(data["Country"].value_counts(dropna=False).head())

In [ ]:
# 3.7
COLUMN_MAP={
    "InvoiceNo": "invoice_no", "StockCode": "stock_code",
    "Description": "description", "Quantity": "quantity",
    "InvoiceDate": "invoice_date", "UnitPrice": "unit_price",
    "CustomerID": "customer_id", "Country": "country",
}
data.rename(columns=COLUMN_MAP, inplace=True)
print(data.columns)


In [ ]:
# 3.10
# Custom ID
print("Number of missing customer IDs:", data["customer_id"].isnull().sum())
data["customer_id"] = data["customer_id"].fillna(-1)
# Empty descriptions
# mask = data["description"].str.strip().eq("")
print("Number of empty descriptions:", data["description"].isnull().sum())
data["description"] = data["description"].fillna(pd.NA)


In [ ]:
# 3.11
data.dropna(subset=["description"])
# The description doesn't help us analyze the data.

In [ ]:
# 3.12
# is_cancelled bad_price bad_qty
is_cancelled = data["invoice_no"].astype(str).str.startswith("C")
bad_qty = data["quantity"] <= 0
bad_price = data["unit_price"] <= 0

print(f"Data size before dropping invalid rows: {len(data)}")
data = data.drop(index=data[is_cancelled | bad_price | bad_qty].index)
not_product = (~data["stock_code"].astype(str).str.match(r"^\d{5}"))
data = data.drop(index=data[not_product].index)
data = data.dropna(subset=["description"])
print(f"Data size after dropping invalid rows: {len(data)}")

In [ ]:
# 3.13
duplicate_number = int(data.duplicated().sum())
data = data.drop_duplicates().reset_index(drop=True)
print("Exact duplicate rows removed:", duplicate_number)
print("Rows remaining:", len(data))

In [ ]:
# Phase D
# 3.18
data.rename(columns={"Revenue": "revenue"}, inplace=True)
# data.columns

data["description"] = data["description"].apply(lambda s: str(s).strip().title())
data["is_cancelled"] = data["invoice_no"].astype(str).str.startswith("C")

data.head()

In [ ]:
# 3.17
import time

s = time.time()
cleaned_desc_list = []
unique_data = data["description"].unique()
for des in unique_data:
    cleaned = str(des).strip().title()
    cleaned_desc_list.append(cleaned)
e = time.time()
print(f"for-loop: time cost in {e - s:.4f} seconds")

s = time.time()
cleaned_desc_list = [
    str(des).strip().title() for des in data["description"].unique()
]
e = time.time()
print(f"list comprehension: time cost in {e - s:.4f} seconds")

s = time.time()
data["description"] = data["description"].apply(lambda s: str(s).strip().title())
e = time.time()
print(f"apply: time cost in {e - s:.4f} seconds")

# In my timing test, approach `list comprehension` > `for-loop` > `apply`, in terms of speed.
# This might be reasonable because apply often still executes Python-level operations element by element.

In [ ]:
# 3.14
rev_by_country = data.groupby("country")["revenue"].sum().sort_values(ascending=False)
print("Revenue by country:", rev_by_country.head(), sep="\n")

print(f"\n{'-' * 25}\n")

orders_per_customer = data[data["customer_id"] != -1].groupby("customer_id")["invoice_no"].nunique().sort_values(ascending=False)
print("Orders per customer:", orders_per_customer.head(), sep="\n")

In [ ]:
# 3.16
by_country = data.groupby("country").agg(
    total_revenue=("revenue", "sum"),
    mean_line_value=("revenue", "mean"),
    transactions=("invoice_no", "nunique"),
).sort_values("total_revenue", ascending=False)
# by_country.head()

mask = data["customer_id"] != -1
by_customer = data[mask].groupby("customer_id").agg(
    total_revenue=("revenue", "sum"),
    mean_line_value=("revenue", "mean"),
    transactions=("invoice_no", "nunique"),
).sort_values("transactions", ascending=False)
by_customer.head()

In [ ]:
# 3.19
data["invoice_date"] = pd.to_datetime(data["invoice_date"])  # Also for 3.15

valid_customers = data[data["customer_id"] != -1]

def customer_summary(cs):
    return pd.Series({
        "total_spend":   cs["revenue"].sum(),
        "n_orders":      cs["invoice_no"].nunique(),
        "active_months": cs["invoice_date"].dt.to_period("M").nunique(),
    })

per_customer = valid_customers.groupby("customer_id").apply(customer_summary)
per_customer["n_orders"] = per_customer["n_orders"].astype(int)
per_customer["active_months"] = per_customer["active_months"].astype(int)
per_customer.sort_values("n_orders", ascending=False).head()


In [ ]:
# 3.15
ts = data.set_index("invoice_date").sort_index()

# Monthly
monthly = ts["revenue"].resample("MS").sum()
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(monthly.index, monthly.values, marker="o")
ax.set_title("Monthly revenue"); ax.set_xlabel("Month"); ax.set_ylabel("Revenue")
fig.autofmt_xdate(); fig.tight_layout()
fig.savefig(r"monthly_revenue.png")

# Weekly
weekly = ts["revenue"].resample("W").sum()
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(weekly.index, weekly.values, marker="o")
ax.set_title("Weekly revenue"); ax.set_xlabel("Week"); ax.set_ylabel("Revenue")
fig.autofmt_xdate(); fig.tight_layout()
fig.savefig(r"weekly_revenue.png")
plt.show()

In [ ]:
# END of Phoenix
df = data

In [ ]:
first_half = df[df["invoice_date"].dt.month <= 6]
second_half = df[df["invoice_date"].dt.month > 6]
pd.concat([first_half, second_half])

In [ ]:
region_lookup_df = pd.DataFrame(list(region_lookup_table.items()), columns=["country", "region"])

df_left = df.merge(region_lookup_df, on="country", how="left")
df_inner = df.merge(region_lookup_df, on="country", how="inner")

display(df_left.head())
display(df_inner.head())

In [ ]:
# Save the cleaned data to a new CSV file
data.to_csv(r"clean_online_retail.csv", index=False)

In [ ]:
# Question
data = pd.read_csv(r"./clean_online_retail.csv")
data["invoice_date"] = pd.to_datetime(data["invoice_date"])
ts = data.set_index("invoice_date").sort_index()

In [ ]:
# Q1: Seasonality. What was total revenue for the year,
# and how does it break down by month?
# When is the peak trading season, and by how much does it lift sales?
total_revenue = ts["revenue"].sum()
monthly: pd.Series = ts["revenue"].resample("MS").sum()

best_season, best_season_revenue = monthly.idxmax(), monthly.max()
lift_percent = best_season_revenue / monthly.mean() - 1

print(f"Total revenue for the year: {total_revenue:.2f}")
print("Monthly revenue breakdown:", monthly.to_period("M"), sep="\n")
print(f"Best season revenue: {best_season_revenue} in {best_season.strftime('%B %Y')}")  # pyright: ignore[reportAttributeAccessIssue]
print(f"Lift percent: {lift_percent:.2%}")

Q1 Interpretation:
Revenue is related to seasonality, with a peak trading season from Septermber to November. Staffing and stock should be weiahted toward this period.

In [ ]:
# Q2
# Q2: Best sellers. Which 10 products earned the most revenue, and which 10 sold the most units?
# Are they the same list and what does any difference tell you about pricing?
# data["stock_code"]
top_revenue = data.groupby("stock_code")["revenue"].sum().nlargest(10)
top_units = data.groupby("stock_code")["quantity"].sum().nlargest(10)

overlap = len(set(top_revenue.index) & set(top_units.index))
print(top_revenue)
print(f"{'-'*25}")
print(top_units)
print(f"{'-'*25}")
print(f"Overlap: {overlap} products are in both lists.")

Q2 Interpretation: There are six items that are in both lists, which means high sales can lead to high revenue.

In [ ]:
# Q3: Markets. Outside the UK, which countries (and which regions, using your merged lookup) are most valuable
# by revenue and by number of distinct customers? Where would you expand?

non_uk = data[data["country"] != "United Kingdom"]
non_uk["region"] = non_uk["country"].map(lambda x: region_lookup_table.get(x))

by_country = non_uk.groupby("country").agg(
    revenue=("revenue", "sum"),
    customers=("customer_id", lambda s: s[s != -1].nunique())
).sort_values("revenue", ascending=False)

by_region = non_uk.groupby("region").agg(
    revenue=("revenue", "sum"),
    customers=("customer_id", lambda s: s[s != -1].nunique())
)

print("Revenue:")
print(by_country.sort_values("revenue", ascending=False).head(3))
print(by_region.sort_values("revenue", ascending=False).head(3))
print(f"{'-'*25}")
print("Distinct customers:")
print(by_country.sort_values("customers", ascending=False).head(3))
print(by_region.sort_values("customers", ascending=False).head(3))

Q3 Interpretation: Western Europe is the most profitable region with the largest number of customers. The company should focus on developing this region to maximize revenue.

In [ ]:
# Q4: Customer concentration. Who are the top 10 customers by total spend,
# and what share of total revenue do the top 1% of customers account for? Is this a wholesale-driven business?

real_customers = data[data["customer_id"] != -1]

customer_revenue = real_customers.groupby("customer_id")["revenue"].sum().sort_values(ascending=False)
top_customers = customer_revenue.head(10)

n_top1 = max(1, int(len(customer_revenue) / 100))
top1pct_revenue = customer_revenue.head(n_top1).sum() / customer_revenue.sum()

print("Top 10 customers by total spend:\n", top_customers)
print(f"\nTop 1% of customers ({n_top1} of {len(customer_revenue)}) account for {top1pct_revenue:.2%} of total revenue")


Q4 Interpretation: The top 10 customers account for around 32% of total revenue, so I think it's a wholesale-driven business.

In [ ]:
# Q5: Order value. What is the average order value (revenue per invoice), and how does it differ between UK and non-UK customers?
order_value = data.groupby("invoice_no").agg(
    revenue=("revenue", "sum"),
    country=("country", "first"),
)

avg_order_value = order_value["revenue"].mean()
mask = order_value["country"] == "United Kingdom"
uk_avg = order_value.loc[mask, "revenue"].mean()
non_uk_avg = order_value.loc[~mask, "revenue"].mean()

print(f"Average order value (all invoices): {avg_order_value:.2f}")
print(f"Average order value (UK): {uk_avg:.2f}")
print(f"Average order value (non-UK): {non_uk_avg:.2f}")


Q5 Interpretation: The average order value for all invoices is approximately 518.22. When we break it down by customer location, we find that UK customers have a lower average order value of about 487.35, while non-UK customers have a significantly higher average order value of around 813.45. 
This suggests that non-UK customers tend to place larger orders compared to UK customers.

In [ ]:
# Q6 Returns & cancellations. How common are cancellations/returns, by count and by value?
# Which products or customers are most associated with them?
# [QUESTION] Use cleaned data?
raw = pd.read_csv(r"./data.csv", encoding="ISO-8859-1")
raw["CustomerID"] = raw["CustomerID"].astype("Int64")

cancels = raw[raw["InvoiceNo"].astype(str).str.startswith("C")]
returns = raw[raw["Quantity"] < 0]

cancel_value = (cancels["Quantity"] * cancels["UnitPrice"]).sum()
cancel_count_ratio = len(cancels) / len(data)

print(f"Cancel value: {cancel_value}, cancel count ratio: {cancel_count_ratio:.2%}")

Q6 Interpretation: The cancel ratio is relatively low at 1.78%, indicating that cancellations are not very common, but the total value of cancellations is significant at -896,812.49, suggesting that when cancellations do occur, they can have a substantial financial impact.

In [ ]:
# Q7: Data-quality memo. What share of the raw rows did you remove or repair (missing IDs, duplicates, cancellations, bad prices)?
# What assumptions did you make, and would you trust this dataset for a board report? Why or why not?
raw = pd.read_csv(r"./data.csv", encoding="ISO-8859-1")

removed = len(raw) - len(data)
print(f"Raw rows: {len(raw):,}")
print(f"Kept as completed sales: {len(data):,} ({len(data)/len(raw):.2%})")
print(f"Removed / repaired: {removed:,} ({removed/len(raw):.2%})")

Q7 Interpretation: The data-quality memo indicates that a small portion of the raw rows (3.58%) were removed or repaired due to issues such as missing IDs, duplicates, cancellations, or bad prices.
I might won't trust this dataset for a board report without further validation, since the data can be forged or manipulated, and the removed rows could contain important information that may affect the overall analysis.